# Installing EdgeR & limma

In [3]:
if (!require("BiocManager", quietly = TRUE)) {
    install.packages("BiocManager", repos = "http://cran.us.r-project.org")
}

# Install edgeR and limma from Bioconductor
BiocManager::install(c("edgeR", "limma"))

Bioconductor version '3.18' is out-of-date; the current release version '3.23'
  is available with R version '4.6'; see https://bioconductor.org/install

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.r-project.org

Bioconductor version 3.18 (BiocManager 1.30.26), R 4.3.3 (2024-02-29)

Installing package(s) 'edgeR', 'limma'

installiere auch Abhängigkeiten ‘locfit’, ‘statmod’


Aktualisiere HTML-Index der Pakete in '.Library'

Making 'packages.html' ...
 fertig

Old packages: 'abind', 'base64enc', 'BH', 'BiocManager', 'bslib', 'callr',
  'cli', 'clipr', 'cluster', 'colorspace', 'cpp11', 'curl', 'data.table',
  'devtools', 'digest', 'downlit', 'dqrng', 'ellipsis', 'fansi', 'fastDummies',
  'fitdistrplus', 'fs', 'future', 'future.apply', 'gert', 'ggplot2', 'ggrepel',
  'gh', 'globals', 'glue', 'gplots', 'gridExtra', 'here', 'highr', 'htmltools',
  'httpuv',

# EdgeR: Each Condition vs Control

In [7]:
# edger for cell_state level

options(warn = -1)
options(stringsAsFactors = FALSE)

library(edgeR)
library(limma)
library(stringr)
library(dplyr)

# configs

base_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_state/Ref_control"
pb_dir   <- file.path(base_dir, "pseudobulks")

out_raw  <- file.path(base_dir, "results/edgeR_raw")
out_sig  <- file.path(base_dir, "results/edgeR_significant")

dir.create(out_raw, recursive = TRUE, showWarnings = FALSE)
dir.create(out_sig, recursive = TRUE, showWarnings = FALSE)

ref_condition <- "Control"
conditions_to_test <- c("3month-protocol", "6month-protocol", "Prematurely-terminated", "PCMV/PRV", "Rejection")

# load clinical info
meta_full <- read.csv(file.path(base_dir, "meta_info.csv"), stringsAsFactors = FALSE)
rownames(meta_full) <- meta_full$Sample_ID
colnames(meta_full)[which(colnames(meta_full) == "Condition")] <- "Disease"


# 2. Main loop

pb_files <- list.files(pb_dir, pattern = "\\.csv$")

for (ct_file in pb_files) {
  # Parse the Cell Type and Cell State from the file names
  filename_no_ext <- gsub("\\.csv", "", ct_file)
  parts <- strsplit(filename_no_ext, "___")[[1]]
  
  if (length(parts) != 2) {
    cat("\nSkipping unrecognized file format:", ct_file, "\n")
    next
  }
  
  celltype <- parts[1]
  cellstate <- parts[2]
  
  cat("\n=========================================================\n")
  cat("PROCESSING CELL TYPE:", celltype, "| CELL STATE:", cellstate, "\n")
  cat("=========================================================\n")
  
  # Load Pseudobulk Matrix
  pesudo_count <- read.csv(file.path(pb_dir, ct_file), check.names = FALSE, row.names = 1)
  
  # Loop over all comparison groups against Ref_condition
  for (cond in conditions_to_test) {
    cat("  -> Comparing:", cond, "vs", ref_condition, "\n")
    
    # 1. Subset metadata for this specific comparison
    sub_meta <- meta_full[meta_full$Disease %in% c(ref_condition, cond), , drop = FALSE]
    
    # 2. Select valid samples
    valid_samples <- intersect(colnames(pesudo_count)[colSums(pesudo_count) > 0], rownames(sub_meta))
    
    if (length(valid_samples) == 0) {
      cat("     SKIP: No valid samples.\n")
      next
    }
    
    mat <- pesudo_count[, valid_samples, drop = FALSE]
    meta <- sub_meta[valid_samples, , drop = FALSE]
    
    # 3. Filter genes (sum > 0 across selected samples)
    mat <- mat[rowSums(mat) > 0, , drop = FALSE]
    
    # 4. Check Group Sizes
    n_ref  <- sum(meta$Disease == ref_condition)
    n_cond <- sum(meta$Disease == cond)
    
    if (n_ref > 2 && n_cond > 2) {
      
      # main edgeR analysis core code
      deglist <- DGEList(counts = mat, group = as.factor(meta$Disease))
      
      # 5. Filter low expression genes
      idx_ref  <- deglist$samples$group == ref_condition
      idx_cond <- deglist$samples$group == cond
      
      mean_ref  <- rowMeans(deglist$counts[, idx_ref, drop = FALSE])
      mean_cond <- rowMeans(deglist$counts[, idx_cond, drop = FALSE])
      
      keep <- which(mean_ref > 0.0125 | mean_cond > 0.0125)
      deglist_keep <- deglist[keep, keep.lib.sizes = FALSE]
      
      # 6. Normalize and model
      deglist_norm <- calcNormFactors(deglist_keep, method = "TMM")
      
      safe_disease <- make.names(meta$Disease)
      design <- model.matrix(~0 + safe_disease)
      colnames(design) <- gsub("safe_disease", "", colnames(design))
      
      deg <- estimateDisp(deglist_norm, design, robust = TRUE)
      fit <- glmQLFit(deg, design)
      
      safe_cond <- make.names(cond)
      safe_ref  <- make.names(ref_condition)
      contrast_str <- paste(safe_cond, safe_ref, sep = "-")
      
      compare <- makeContrasts(contrasts = contrast_str, levels = design)
      qlf <- glmQLFTest(fit, contrast = compare)
      
      # 7. Extract results
      res <- topTags(qlf, n = nrow(deglist_norm$counts))$table
      
      res[, paste0("Observations_", cond)] <- n_cond
      res[, paste0("Observations_", ref_condition)] <- n_ref
      
      # Add Metadata tracking both CellType and CellState
      res$Gene <- rownames(res)
      res$CellType <- celltype
      res$CellState <- cellstate
      res$Comparison <- paste(cond, "vs", ref_condition)
      
      res <- res[, c("Gene", "CellType", "CellState", "Comparison", "logFC", "logCPM", "F", "PValue", "FDR", 
                     paste0("Observations_", cond), paste0("Observations_", ref_condition))]
      
      # organize directories
      comp_folder_name <- paste0(gsub("[^A-Za-z0-9_]", "_", cond), "_vs_", ref_condition)
      
      # Export raw results - Added `celltype` nested subfolder
      comp_raw_dir <- file.path(out_raw, comp_folder_name, celltype)
      dir.create(comp_raw_dir, recursive = TRUE, showWarnings = FALSE)
      
      raw_filename <- paste0("edgeR_Raw_", cellstate, "_", comp_folder_name, ".csv")
      write.csv(res, file.path(comp_raw_dir, raw_filename), row.names = FALSE)
      
      # Filter, sort & export significant results
      res_sig <- res[res$FDR < 0.05 & abs(res$logFC) >= 0.5, ]
      
      if (nrow(res_sig) > 0) {
        res_sig$type <- ifelse(res_sig$logFC > 0, "up", "down")
        res_sig <- res_sig[order(res_sig$logFC, decreasing = TRUE), ]
        
        res_sig <- res_sig[, c("Gene", "CellType", "CellState", "Comparison", "type", "logFC", "logCPM", "F", "PValue", "FDR", 
                               paste0("Observations_", cond), paste0("Observations_", ref_condition))]
        
        # Export significant results
        comp_sig_dir <- file.path(out_sig, comp_folder_name, celltype)
        dir.create(comp_sig_dir, recursive = TRUE, showWarnings = FALSE)
        
        sig_filename <- paste0("edgeR_Sig_", cellstate, "_", comp_folder_name, ".csv")
        write.csv(res_sig, file.path(comp_sig_dir, sig_filename), row.names = FALSE)
        cat("     -> Found", nrow(res_sig), "significant DEGs (sorted and saved in", celltype, "subfolder).\n")
      } else {
        cat("     -> No significant DEGs passed the threshold (FDR < 0.05, |log2FC| >= 0.5).\n")
      }
      
    } else {
      cat("     SKIP: Insufficient samples. Ref (", n_ref, "), Cond (", n_cond, ") - Both must be > 2.\n")
    }
  }
}

cat("\nAll edgeR pipelines complete! Check the results/edgeR_raw and results/edgeR_significant folders.\n")


PROCESSING CELL TYPE: CM | CELL STATE: CM1 
  -> Comparing: 3month-protocol vs Control 
     -> Found 4088 significant DEGs (sorted and saved in CM subfolder).
  -> Comparing: 6month-protocol vs Control 
     -> Found 3150 significant DEGs (sorted and saved in CM subfolder).
  -> Comparing: Prematurely-terminated vs Control 
     -> Found 1555 significant DEGs (sorted and saved in CM subfolder).
  -> Comparing: PCMV/PRV vs Control 
     -> Found 2295 significant DEGs (sorted and saved in CM subfolder).
  -> Comparing: Rejection vs Control 
     -> Found 2869 significant DEGs (sorted and saved in CM subfolder).

PROCESSING CELL TYPE: CM | CELL STATE: CM2 
  -> Comparing: 3month-protocol vs Control 
     -> Found 3324 significant DEGs (sorted and saved in CM subfolder).
  -> Comparing: 6month-protocol vs Control 
     -> Found 2907 significant DEGs (sorted and saved in CM subfolder).
  -> Comparing: Prematurely-terminated vs Control 
     -> Found 613 significant DEGs (sorted and saved 

# EdgeR: all xHTx vs Control

In [1]:
# EdgeR on cell_state level: supergrouping: all xenografts vs controls pipeline

options(warn = -1)
options(stringsAsFactors = FALSE)

# Explicitly load packages
library(edgeR)
library(limma)
library(stringr)
library(dplyr)

# Configs and paths
base_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_state/Ref_control"
pb_dir   <- file.path(base_dir, "pseudobulks")

out_raw  <- file.path(base_dir, "results/edgeR_raw")
out_sig  <- file.path(base_dir, "results/edgeR_significant")

dir.create(out_raw, recursive = TRUE, showWarnings = FALSE)
dir.create(out_sig, recursive = TRUE, showWarnings = FALSE)

ref_condition <- "Control"
pooled_cond_name <- "xHTx"

# Load clinical info
meta_full <- read.csv(file.path(base_dir, "meta_info.csv"), stringsAsFactors = FALSE)
rownames(meta_full) <- meta_full$Sample_ID

colnames(meta_full)[which(colnames(meta_full) == "Condition")] <- "Disease"

# pooling logic
meta_full$Pooled_Condition <- ifelse(meta_full$Disease == ref_condition, ref_condition, pooled_cond_name)

# main loop using all cell states
pb_files <- list.files(pb_dir, pattern = "\\.csv$")

for (ct_file in pb_files) {
  filename_no_ext <- gsub("\\.csv", "", ct_file)
  parts <- strsplit(filename_no_ext, "___")[[1]]
  
  if (length(parts) != 2) {
    cat("\nSkipping unrecognized file format:", ct_file, "\n")
    next
  }
  
  celltype <- parts[1]
  cellstate <- parts[2]
  
  cat("\n=========================================================\n")
  cat("PROCESSING CELL TYPE:", celltype, "| CELL STATE:", cellstate, "\n")
  cat("=========================================================\n")
  
  # Load Pseudobulk Matrix
  pesudo_count <- read.csv(file.path(pb_dir, ct_file), check.names = FALSE, row.names = 1)
  
  cat("  -> Comparing:", pooled_cond_name, "vs", ref_condition, "\n")
  
  # 1. Subset metadata for valid samples (all samples should technically be valid here)
  sub_meta <- meta_full
  
  # 2. Select valid samples (colSums > 0 and present in sub_meta)
  valid_samples <- intersect(colnames(pesudo_count)[colSums(pesudo_count) > 0], rownames(sub_meta))
  
  if (length(valid_samples) == 0) {
    cat("     SKIP: No valid samples.\n")
    next
  }
  
  mat <- pesudo_count[, valid_samples, drop = FALSE]
  meta <- sub_meta[valid_samples, , drop = FALSE]
  
  # 3. Filter genes (sum > 0 across selected samples)
  mat <- mat[rowSums(mat) > 0, , drop = FALSE]
  
  # 4. Check Group Sizes
  n_ref  <- sum(meta$Pooled_Condition == ref_condition)
  n_cond <- sum(meta$Pooled_Condition == pooled_cond_name)
  
  if (n_ref > 2 && n_cond > 2) {
    
    # edgeR main analysis core codes
    
    # Group using the newly pooled condition
    deglist <- DGEList(counts = mat, group = as.factor(meta$Pooled_Condition))
    
    # 5. Filter low expression genes (mean > 0.0125 in AT LEAST ONE group)
    idx_ref  <- deglist$samples$group == ref_condition
    idx_cond <- deglist$samples$group == pooled_cond_name
    
    mean_ref  <- rowMeans(deglist$counts[, idx_ref, drop = FALSE])
    mean_cond <- rowMeans(deglist$counts[, idx_cond, drop = FALSE])
    
    keep <- which(mean_ref > 0.0125 | mean_cond > 0.0125)
    deglist_keep <- deglist[keep, keep.lib.sizes = FALSE]
    
    # 6. Normalize and Model
    deglist_norm <- calcNormFactors(deglist_keep, method = "TMM")
    
    # Safe names for modeling
    safe_disease <- make.names(meta$Pooled_Condition)
    design <- model.matrix(~0 + safe_disease)
    colnames(design) <- gsub("safe_disease", "", colnames(design))
    
    deg <- estimateDisp(deglist_norm, design, robust = TRUE)
    fit <- glmQLFit(deg, design)
    
    # Build Contrast
    safe_cond <- make.names(pooled_cond_name)
    safe_ref  <- make.names(ref_condition)
    contrast_str <- paste(safe_cond, safe_ref, sep = "-")
    
    compare <- makeContrasts(contrasts = contrast_str, levels = design)
    qlf <- glmQLFTest(fit, contrast = compare)
    
    # 7. Extract results
    res <- topTags(qlf, n = nrow(deglist_norm$counts))$table
    
    # Add observation counts
    res[, paste0("Observations_", pooled_cond_name)] <- n_cond
    res[, paste0("Observations_", ref_condition)] <- n_ref
    
    # Add Metadata (Updated for Cell State)
    res$Gene <- rownames(res)
    res$CellType <- celltype
    res$CellState <- cellstate
    res$Comparison <- paste(pooled_cond_name, "vs", ref_condition)
    
    # Reorder columns
    res <- res[, c("Gene", "CellType", "CellState", "Comparison", "logFC", "logCPM", "F", "PValue", "FDR", 
                   paste0("Observations_", pooled_cond_name), paste0("Observations_", ref_condition))]
    

    # ORGANIZE DIRECTORIES AND EXPORT 
    
    comp_folder_name <- paste0(pooled_cond_name, "_vs_", ref_condition)
    
    # Export Raw Results - Nested by celltype
    comp_raw_dir <- file.path(out_raw, comp_folder_name, celltype)
    dir.create(comp_raw_dir, recursive = TRUE, showWarnings = FALSE)
    
    raw_filename <- paste0("edgeR_Raw_", cellstate, "_", comp_folder_name, ".csv")
    write.csv(res, file.path(comp_raw_dir, raw_filename), row.names = FALSE)
    

    # FILTER, SORT & EXPORT SIGNIFICANT RESULTS

    # Apply FDR < 0.05 and |logFC| >= 0.5
    res_sig <- res[res$FDR < 0.05 & abs(res$logFC) >= 0.5, ]
    
    if (nrow(res_sig) > 0) {
      
      # 1. Add 'type' column
      res_sig$type <- ifelse(res_sig$logFC > 0, "up", "down")
      
      # 2. Sort descending by logFC
      res_sig <- res_sig[order(res_sig$logFC, decreasing = TRUE), ]
      
      # 3. Reorder to show 'type'
      res_sig <- res_sig[, c("Gene", "CellType", "CellState", "Comparison", "type", "logFC", "logCPM", "F", "PValue", "FDR", 
                             paste0("Observations_", pooled_cond_name), paste0("Observations_", ref_condition))]
      
      # 4. Save; nested by celltype
      comp_sig_dir <- file.path(out_sig, comp_folder_name, celltype)
      dir.create(comp_sig_dir, recursive = TRUE, showWarnings = FALSE)
      
      sig_filename <- paste0("edgeR_Sig_", cellstate, "_", comp_folder_name, ".csv")
      write.csv(res_sig, file.path(comp_sig_dir, sig_filename), row.names = FALSE)
      cat("     -> Found", nrow(res_sig), "significant DEGs (sorted and saved in", celltype, "subfolder).\n")
    } else {
      cat("     -> No significant DEGs passed the threshold (FDR < 0.05, |log2FC| >= 0.5).\n")
    }
    
  } else {
    cat("     SKIP: Insufficient samples. Ref (", n_ref, "), Cond (", n_cond, ") - Both must be > 2.\n")
  }
}

cat("\nPooled edgeR pipeline complete! Check the xHTx_vs_Control folders in results/edgeR_raw and results/edgeR_significant.\n")

Lade nötiges Paket: limma




Attache Paket: ‘dplyr’




Die folgenden Objekte sind maskiert von ‘package:stats’:

    filter, lag




Die folgenden Objekte sind maskiert von ‘package:base’:

    intersect, setdiff, setequal, union





PROCESSING CELL TYPE: CM | CELL STATE: CM1 
  -> Comparing: xHTx vs Control 
     -> Found 3676 significant DEGs (sorted and saved in CM subfolder).

PROCESSING CELL TYPE: CM | CELL STATE: CM2 
  -> Comparing: xHTx vs Control 
     -> Found 2832 significant DEGs (sorted and saved in CM subfolder).

PROCESSING CELL TYPE: CM | CELL STATE: CM3 
  -> Comparing: xHTx vs Control 
     -> Found 2483 significant DEGs (sorted and saved in CM subfolder).

PROCESSING CELL TYPE: CM | CELL STATE: CM4 
  -> Comparing: xHTx vs Control 
     -> Found 2505 significant DEGs (sorted and saved in CM subfolder).

PROCESSING CELL TYPE: CM | CELL STATE: CM5 
  -> Comparing: xHTx vs Control 
     -> Found 13 significant DEGs (sorted and saved in CM subfolder).

PROCESSING CELL TYPE: CM | CELL STATE: Hypoxia_CM 
  -> Comparing: xHTx vs Control 
     -> Found 3304 significant DEGs (sorted and saved in CM subfolder).

PROCESSING CELL TYPE: CM | CELL STATE: ISG_CM 
  -> Comparing: xHTx vs Control 
     -> Found 


Pooled edgeR pipeline complete! Check the xHTx_vs_Control folders in results/edgeR_raw and results/edgeR_significant.


# overall barplot

In [2]:
# DEG summaries and barplots

options(warn = -1)
options(stringsAsFactors = FALSE)

# Load required packages
quiet_library <- function(pkg) {
  suppressWarnings(suppressMessages(require(pkg, character.only = TRUE)))
}

pkgs <- c("dplyr", "ggplot2", "tidyr", "stringr")
invisible(lapply(pkgs, quiet_library))

# config, path to where the signficant results are at
base_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_state/Ref_control"
sig_dir  <- file.path(base_dir, "results/edgeR_significant")

# Output directory for the master CSV and barplots
out_dir <- file.path(base_dir, "results/overall_barplots")
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

# Custom plot theme (Arial/sans, Size 5, Bold, No Grids)
custom_theme <- theme(
  text = element_text(size = 5, face = "bold", family = "sans", color = "black"),
  axis.text.x = element_text(angle = 45, hjust = 1, face = "bold", size = 5, family = "sans", color = "black"),
  axis.text.y = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  axis.title = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  plot.title = element_text(face = "bold", size = 5, family = "sans", color = "black", hjust = 0.5),
  legend.text = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  legend.title = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  panel.background = element_blank(),
  panel.grid.major = element_blank(),
  panel.grid.minor = element_blank(),
  axis.line = element_line(color = "black", linewidth = 0.3)
)

# aggregate signficant DEG reuslts
cat("Scanning for significant DEG CSVs...\n")

#  find all CSVs in the edgeR_significant directory
sig_files <- list.files(sig_dir, pattern = "\\.csv$", recursive = TRUE, full.names = TRUE)

if (length(sig_files) == 0) {
  stop("No significant CSV files found in the specified directory.")
}

summary_list <- list()

for (f in sig_files) {
  df <- read.csv(f, stringsAsFactors = FALSE)
  
  if (nrow(df) == 0) next
  
  # Extract comparison, cell type, and cell state from the dataframe
  comp   <- df$Comparison[1]
  ctype  <- df$CellType[1]
  cstate <- df$CellState[1]
  
  # Count occurrences
  n_total <- nrow(df)
  n_up    <- sum(df$type == "up", na.rm = TRUE)
  n_down  <- sum(df$type == "down", na.rm = TRUE)
  
  summary_list[[length(summary_list) + 1]] <- data.frame(
    Comparison = comp,
    Cell_Type = ctype,
    Cell_State = cstate,
    Total_Significant_DEGs = n_total,
    Up_DEGs = n_up,
    Down_DEGs = n_down,
    stringsAsFactors = FALSE
  )
}

# Combine all into a master dataframe
master_df <- dplyr::bind_rows(summary_list)

# Save Master CSV
master_csv_path <- file.path(out_dir, "WholeMaster_DEG_Counts.csv")
write.csv(master_df, master_csv_path, row.names = FALSE)
cat("Master CSV saved to:", master_csv_path, "\n")

# 3. GENERATE BARPLOTS PER COMPARISON & CELL TYPE

cat("Generating barplots...\n")

# Convert dataframe to long format for plotting
plot_df <- master_df %>%
  tidyr::pivot_longer(cols = c("Up_DEGs", "Down_DEGs"), 
                      names_to = "Direction", 
                      values_to = "Count") %>%
  dplyr::mutate(Direction = ifelse(Direction == "Up_DEGs", "Upregulated", "Downregulated"))

# Ensure upregulated plots first (left), then downregulated (right)
plot_df$Direction <- factor(plot_df$Direction, levels = c("Upregulated", "Downregulated"))

unique_comps <- unique(plot_df$Comparison)

for (comp in unique_comps) {
  
  # Subset data for the specific comparison
  comp_df <- plot_df %>% filter(Comparison == comp)
  unique_ctypes <- unique(comp_df$Cell_Type)
  
  # Loop through each Cell Type to generate a separate plot showing its Cell States
  for (ctype in unique_ctypes) {
    
    sub_df <- comp_df %>% filter(Cell_Type == ctype)
    
    # Plot
    p <- ggplot(sub_df, aes(x = Cell_State, y = Count, fill = Direction)) +
      geom_bar(stat = "identity", position = position_dodge(width = 0.8), width = 0.7, color = "black", linewidth = 0.15) +
      scale_fill_manual(values = c("Upregulated" = "salmon", "Downregulated" = "skyblue")) +
      # Add text labels right above the bars
      geom_text(aes(label = Count), 
                position = position_dodge(width = 0.8), 
                vjust = -0.5, 
                size = 5 / .pt, fontface = "bold", family = "sans", color = "black") +
      labs(title = paste(comp, "-", ctype), 
           x = "Cell State", 
           y = "Number of Significant DEGs", 
           fill = "Regulation") +
      scale_y_continuous(expand = expansion(mult = c(0, 0.2))) + # Extra room at the top for labels
      custom_theme
    
    # Generate safe filenames
    safe_comp_name <- gsub("[^A-Za-z0-9_]", "_", comp)
    safe_ctype_name <- gsub("[^A-Za-z0-9_]", "_", ctype)
    
    # Optional: Create comparison subfolders inside overall_barplots to keep things tidy
    comp_out_dir <- file.path(out_dir, safe_comp_name)
    dir.create(comp_out_dir, recursive = TRUE, showWarnings = FALSE)
    
    plot_filename <- file.path(comp_out_dir, paste0("DEG_Barplot_", safe_comp_name, "_", safe_ctype_name, ".pdf"))
    
    # Save to PDF
    ggsave(plot_filename, plot = p, width = 4, height = 2.5, units = "in", dpi = 300)
    cat("  -> Saved plot for:", comp, "|", ctype, "\n")
  }
}

cat("All summaries and barplots completed successfully!\n")

Scanning for significant DEG CSVs...


Master CSV saved to: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_state/Ref_control/results/overall_barplots/WholeMaster_DEG_Counts.csv 


Generating barplots...


  -> Saved plot for: 3month-protocol vs Control | CM 


  -> Saved plot for: 3month-protocol vs Control | EC 


  -> Saved plot for: 3month-protocol vs Control | FB 


... (18 outputs left out)

  -> Saved plot for: xHTx vs Control | FB 


  -> Saved plot for: xHTx vs Control | Lymph_sus 


  -> Saved plot for: xHTx vs Control | Mye_sus 


All summaries and barplots completed successfully!


# Volcano plots: Unlabeled

In [10]:
# generation of volcano plots without gene labels

options(warn = -1)
options(stringsAsFactors = FALSE)

quiet_library <- function(pkg) {
  suppressWarnings(suppressMessages(require(pkg, character.only = TRUE)))
}

pkgs <- c("dplyr", "ggplot2", "stringr")
invisible(lapply(pkgs, quiet_library))

# configs
base_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_state/Ref_control"
raw_dir  <- file.path(base_dir, "results/edgeR_raw")

# Base output directory for the volcano plots
out_dir <- file.path(base_dir, "results/volcano_unnamed")
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

# Thresholds 
fdr_threshold   <- 0.05
logfc_threshold <- 0.5

# Custom theme (Arial/sans, Size 5, Bold, No Grids)
custom_theme <- theme(
  text = element_text(size = 5, face = "bold", family = "sans", color = "black"),
  axis.text.x = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  axis.text.y = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  axis.title = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  plot.title = element_text(face = "bold", size = 5, family = "sans", color = "black", hjust = 0.5),
  legend.text = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  legend.title = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  panel.background = element_blank(),
  panel.grid.major = element_blank(),
  panel.grid.minor = element_blank(),
  axis.line = element_line(color = "black", linewidth = 0.3)
)

# generating the volcano plots
cat("Scanning for raw edgeR CSVs...\n")

# Find all raw CSV files (recursive ensures we look inside the comparison/celltype subfolders)
raw_files <- list.files(raw_dir, pattern = "\\.csv$", recursive = TRUE, full.names = TRUE)

if (length(raw_files) == 0) {
  stop("No raw CSV files found in the specified directory.")
}

for (f in raw_files) {
  df <- read.csv(f, stringsAsFactors = FALSE)
  
  if (nrow(df) == 0) next
  
  # Extract metadata from dataframe
  comp   <- df$Comparison[1]
  ctype  <- df$CellType[1]
  cstate <- df$CellState[1]
  
  # Create safe folder and file names
  safe_comp   <- gsub("[^A-Za-z0-9_]", "_", comp)
  safe_ctype  <- gsub("[^A-Za-z0-9_]", "_", ctype)
  safe_cstate <- gsub("[^A-Za-z0-9_]", "_", cstate)
  
  # Create a specific nested subfolder for this comparison AND cell type
  comp_out_dir <- file.path(out_dir, safe_comp, safe_ctype)
  dir.create(comp_out_dir, recursive = TRUE, showWarnings = FALSE)
  
  # 1. Handle extreme P-values/FDRs (prevent infinity when taking log10)
  min_fdr <- min(df$FDR[df$FDR > 0], na.rm = TRUE)
  if (is.na(min_fdr) || min_fdr == Inf) min_fdr <- 1e-300
  df$FDR[df$FDR == 0] <- min_fdr
  
  # Calculate -log10(FDR) for the y axis
  df$neg_log10_fdr <- -log10(df$FDR)
  
  # 2. Define Significance Categories
  df <- df %>%
    mutate(
      Significance = case_when(
        FDR < fdr_threshold & logFC >= logfc_threshold ~ "Upregulated",
        FDR < fdr_threshold & logFC <= -logfc_threshold ~ "Downregulated",
        TRUE ~ "Not Significant"
      )
    )
  
  # Reorder factor so "Not Significant" is plotted first
  df$Significance <- factor(df$Significance, levels = c("Not Significant", "Downregulated", "Upregulated"))
  
  # 3. Create the Volcano Plot
  p <- ggplot(df, aes(x = logFC, y = neg_log10_fdr, color = Significance)) +
    geom_point(alpha = 0.8, size = 0.5, stroke = 0) +
    scale_color_manual(values = c("Not Significant" = "grey80", 
                                  "Downregulated" = "royalblue", 
                                  "Upregulated" = "salmon")) +
    # Add threshold lines
    geom_vline(xintercept = c(-logfc_threshold, logfc_threshold), linetype = "dashed", color = "black", linewidth = 0.2) +
    geom_hline(yintercept = -log10(fdr_threshold), linetype = "dashed", color = "black", linewidth = 0.2) +
    labs(
      # Updated title to include Cell State
      title = paste0(ctype, " | ", cstate, " (", comp, ")"),
      x = bquote(bold(log[2]("Fold Change"))),
      y = bquote(bold(-log[10]("FDR"))),
      color = "Regulation"
    ) +
    custom_theme +
    # Override legend point size so it's visible even though plot points are tiny
    guides(color = guide_legend(override.aes = list(size = 2)))
  
  # ---------------------------------------------------------------------------
  # FULL FRAME TOGGLE:
  # To add a full box frame around the plot, simply remove the '#' at the 
  # start of the line below
  # ---------------------------------------------------------------------------
  # p <- p + theme(panel.border = element_rect(colour = "black", fill = NA, linewidth = 0.3), axis.line = element_blank())

  # 4. Save the plot into the nested celltype subfolder
  plot_filename <- file.path(comp_out_dir, paste0("Volcano_", safe_cstate, "_", safe_comp, ".pdf"))
  
  ggsave(plot_filename, plot = p, width = 2.75, height = 2.25, units = "in", dpi = 300)
  cat("  -> Saved volcano plot for:", cstate, "in folder:", paste(safe_comp, safe_ctype, sep="/"), "\n")
}

cat("\nAll volcano plots generated and safely organized into nested subfolders at:\n", out_dir, "\n")

Scanning for raw edgeR CSVs...
  -> Saved volcano plot for: CM1 in folder: 3month_protocol_vs_Control/CM 
  -> Saved volcano plot for: CM2 in folder: 3month_protocol_vs_Control/CM 
  -> Saved volcano plot for: CM3 in folder: 3month_protocol_vs_Control/CM 
  -> Saved volcano plot for: CM4 in folder: 3month_protocol_vs_Control/CM 
  -> Saved volcano plot for: CM5 in folder: 3month_protocol_vs_Control/CM 
  -> Saved volcano plot for: Hypoxia_CM in folder: 3month_protocol_vs_Control/CM 
  -> Saved volcano plot for: ISG_CM in folder: 3month_protocol_vs_Control/CM 
  -> Saved volcano plot for: Arterial_EC in folder: 3month_protocol_vs_Control/EC 
  -> Saved volcano plot for: Cap_EC in folder: 3month_protocol_vs_Control/EC 
  -> Saved volcano plot for: Hypoxia_EC in folder: 3month_protocol_vs_Control/EC 
  -> Saved volcano plot for: ISG_cap_EC in folder: 3month_protocol_vs_Control/EC 
  -> Saved volcano plot for: Lymph_EC in folder: 3month_protocol_vs_Control/EC 
  -> Saved volcano plot for: 

# Volcano plots: labelled

In [11]:
# generation of volcano plots with some gene labels

options(warn = -1)
options(stringsAsFactors = FALSE)

# Safely load required libraries
quiet_library <- function(pkg) {
  suppressWarnings(suppressMessages(require(pkg, character.only = TRUE)))
}

pkgs <- c("dplyr", "ggplot2", "stringr", "RColorBrewer")
invisible(lapply(pkgs, quiet_library))

# configs
# Input/Output paths

base_raw_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_state/Ref_control/results/edgeR_raw"
out_dir      <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_state/Ref_control/results/volcano_named"

# Experiment details and comparison 
# EXACT name of the folder/comparison in your raw results directory
comp_name         <- "PCMV_PRV_vs_Control"

# The specific cell type and cell state to extract and plot
target_cell_type  <- "EC"
target_cell_state <- "Cap_EC"

# Statistical thresholds
fdr_threshold   <- 0.05
logfc_threshold <- 0.5    # Note: Adjusted to 0.5 per your script

# Genes to visualize
# Add or remove any gene names here. The script will automatically skip missing ones.
genes_to_highlight <- c(
  "Sus_scrofa___MX1", "Sus_scrofa___MX2", "Sus_scrofa___ISG20", "Sus_scrofa___CXCL10", 
  "Sus_scrofa___IFIT1", "Sus_scrofa___OASL", 
  "Sus_scrofa___TSPAN18", "Sus_scrofa___VWF", "Sus_scrofa___SERPINE1"
)


# data preparation and visualization

# Create safe directory and file names
safe_comp   <- gsub("[^A-Za-z0-9_\\-]", "_", comp_name) # Preserves hyphens
safe_ctype  <- gsub("[^A-Za-z0-9_]", "_", target_cell_type)
safe_cstate <- gsub("[^A-Za-z0-9_]", "_", target_cell_state)

# Create  output directory: out_dir/comparison/celltype
comp_out_dir <- file.path(out_dir, comp_name, safe_ctype) 
dir.create(comp_out_dir, recursive = TRUE, showWarnings = FALSE)

# Construct expected raw edgeR CSV path (nested inside celltype subfolder)
raw_file_path <- file.path(base_raw_dir, comp_name, safe_ctype, paste0("edgeR_Raw_", safe_cstate, "_", safe_comp, ".csv"))

if (!file.exists(raw_file_path)) {
  stop(paste("ERROR: Cannot find raw CSV file at:\n", raw_file_path, 
             "\nPlease verify your 'base_raw_dir', 'comp_name', 'target_cell_type', and 'target_cell_state'."))
}

df <- read.csv(raw_file_path, stringsAsFactors = FALSE)

# Validate which highlight genes actually exist in the current dataset
actual_highlights <- intersect(genes_to_highlight, df$Gene)
missing_genes <- setdiff(genes_to_highlight, df$Gene)

if(length(missing_genes) > 0) {
  cat("WARNING: The following requested genes were NOT found in this dataset and will be skipped:\n", 
      paste(missing_genes, collapse = ", "), "\n\n")
}

# Handle extreme pvalues/FDRs
min_fdr <- min(df$FDR[df$FDR > 0], na.rm = TRUE)
if (is.na(min_fdr) || min_fdr == Inf) min_fdr <- 1e-300
df$FDR[df$FDR == 0] <- min_fdr
df$neg_log10_fdr <- -log10(df$FDR)

# Categorize background genes by significance thresholds
df <- df %>%
  mutate(
    Significance = case_when(
      FDR < fdr_threshold & logFC >= logfc_threshold ~ "Upregulated",
      FDR < fdr_threshold & logFC <= -logfc_threshold ~ "Downregulated",
      TRUE ~ "Not Significant"
    )
  )

# Enforce plotting order (Not Significant drawn first in background)
df$Significance <- factor(df$Significance, levels = c("Not Significant", "Downregulated", "Upregulated"))

# Split dataframe: background vs foreground (highlighted)
df_bg <- df[!df$Gene %in% actual_highlights, ]
df_fg <- df[df$Gene %in% actual_highlights, ]

# Order foreground legend alphabetically
if(nrow(df_fg) > 0) {
  df_fg$Gene <- factor(df_fg$Gene, levels = sort(unique(df_fg$Gene)))
}


# 3. visualization params

# Structured, grid-free publication theme (Arial, Size 5, Bold)
custom_theme <- theme(
  text = element_text(size = 5, face = "bold", family = "sans", color = "black"),
  axis.text.x = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  axis.text.y = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  axis.title = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  plot.title = element_text(face = "bold", size = 5, family = "sans", color = "black", hjust = 0.5),
  legend.text = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  legend.title = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  panel.background = element_blank(),
  panel.grid.major = element_blank(),
  panel.grid.minor = element_blank(),
  panel.border = element_rect(colour = "black", fill = NA, linewidth = 0.3), # Full frame
  axis.line = element_blank(), # Suppress standard axis lines due to full frame
  legend.key.size = unit(0.3, "cm")
)

# A color Palette (Supports up to 50 highlighted genes)
prof_colors <- c(
  "#FFD700", "#808080", "#87CEFA", "#800080", "#E41A1C", "#377EB8", "#4DAF4A", "#FF7F00", "#A65628", "#F781BF", 
  "#1B9E77", "#D95F02", "#7570B3", "#E7298A", "#66A61E", "#E6AB02", "#1F78B4", "#B2DF8A", "#33A02C", "#FB9A99", 
  "#FDBF6F", "#CAB2D6", "#6A3D9A", "#FFFF99", "#BEBADA", "#FB8072", "#80B1D3", "#B3DE69", "#D9D9D9", "#BC80BD", 
  "#000000", "#1ce6ff", "#ff34ff", "#008941", "#006fa6", "#a30059", "#7a4900", "#0000a6", "#63ffac", "#b79762", 
  "#004d43"
)

# Expand palette dynamically if user supplies >41 genes
if(length(actual_highlights) > length(prof_colors)) {
  prof_colors <- colorRampPalette(prof_colors)(length(actual_highlights))
}

# plot creation

p <- ggplot() +
  
  # LAYER 1: Background genes (No borders, mapped to significance)
  geom_point(data = df_bg, aes(x = logFC, y = neg_log10_fdr, color = Significance), 
             alpha = 0.8, size = 0.5, stroke = 0) +
  scale_color_manual(values = c("Not Significant" = "grey80", 
                                "Downregulated" = "royalblue", 
                                "Upregulated" = "salmon")) +
  
  # LAYER 2: Highlighted genes (Shape 21 for fill + black border)
  geom_point(data = df_fg, aes(x = logFC, y = neg_log10_fdr, fill = Gene), 
             shape = 21, color = "black", stroke = 0.2, size = 0.75) +
  scale_fill_manual(values = prof_colors) +
  
  # LAYER 3: Threshold lines
  geom_vline(xintercept = c(-logfc_threshold, logfc_threshold), linetype = "dashed", color = "black", linewidth = 0.2) +
  geom_hline(yintercept = -log10(fdr_threshold), linetype = "dashed", color = "black", linewidth = 0.2) +
  
  # LAYER 4: Formatting & Labels (Updated to show Cell State)
  labs(
    title = paste0(target_cell_type, " | ", target_cell_state, " (", comp_name, ")"),
    x = bquote(bold(log[2]("Fold Change"))),
    y = bquote(bold(-log[10]("FDR"))),
    color = "Regulation",
    fill = "Highlighted Genes"
  ) +
  custom_theme +
  
  # Force legend dots to be visually distinct regardless of plot point size
  guides(
    color = guide_legend(override.aes = list(size = 2), order = 1),
    fill  = guide_legend(override.aes = list(size = 2), order = 2)
  )

# Export

plot_filename <- file.path(comp_out_dir, paste0("Volcano_Highlighted_", safe_cstate, "_", safe_comp, ".pdf"))

# Width set to 2.75 inches; expand if legend gets cut off with many genes
ggsave(plot_filename, plot = p, width = 2.75, height = 2, units = "in", dpi = 300)

cat("Successfully generated targeted volcano plot!\nSaved to:", plot_filename, "\n")

Successfully generated targeted volcano plot!
Saved to: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_state/Ref_control/results/volcano_named/PCMV_PRV_vs_Control/EC/Volcano_Highlighted_Cap_EC_PCMV_PRV_vs_Control.pdf 
